# BrandPulse AI — запуск MVP в Google Colab

Выполняйте ячейки сверху вниз. Для демонстрации без YouTube API используйте демоданные или импорт CSV. Приложение хранит SQLite на Google Drive, если Drive подключён. Код `app.py` берётся из комплекта сдачи, без ручных исправлений.


## 1. Загрузите ZIP комплекта проекта из Git-репозитория
Архив должен содержать папку `brandpulse_colab` с `app.py` и `requirements.txt`.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('Загрузите ZIP-архив brandpulse_colab.zip из репозитория')
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content')
assert Path('/content/brandpulse_colab/app.py').exists()
print('Файлы проекта распакованы')


## 2. Установите зависимости и подключите Google Drive

In [ ]:
!pip -q install -r /content/brandpulse_colab/requirements.txt
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
Path('/content/drive/MyDrive/BrandPulseAI').mkdir(parents=True, exist_ok=True)
os.environ['BRANDPULSE_DB_PATH'] = '/content/drive/MyDrive/BrandPulseAI/brandpulse.db'
print('База:', os.environ['BRANDPULSE_DB_PATH'])


## 3. Необязательный доступ к YouTube API
В Colab Secrets добавьте `YOUTUBE_API_KEY`. Без него работает импорт CSV и демоданные.

In [ ]:
from google.colab import userdata
try:
    key = userdata.get('YOUTUBE_API_KEY')
    if key:
        os.environ['YOUTUBE_API_KEY'] = key
        print('Ключ YouTube подключён')
except Exception:
    print('Ключ не задан; используйте демоданные или CSV')


## 4. Запустите Streamlit

In [ ]:
import subprocess, time
from pathlib import Path
log = open('/content/brandpulse_streamlit.log', 'w')
process = subprocess.Popen(['streamlit','run','/content/brandpulse_colab/app.py','--server.port','8501','--server.address','0.0.0.0','--server.headless','true'],stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())
time.sleep(6)
if process.poll() is not None:
    raise RuntimeError(Path('/content/brandpulse_streamlit.log').read_text()[-2000:])
print('Streamlit PID:', process.pid)


## 5. Откройте приложение через временный Cloudflare Tunnel
Временная ссылка изменяется при перезапуске Colab. Для проверки работы не нужен платный сервис.

In [ ]:
import re, urllib.request, stat
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists() or cloudflared.stat().st_size == 0:
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IXUSR)
log_tunnel = open('/content/brandpulse_tunnel.log','w')
tunnel = subprocess.Popen([str(cloudflared),'tunnel','--url','http://127.0.0.1:8501','--no-autoupdate'],stdout=log_tunnel,stderr=subprocess.STDOUT)
public_url = None
for _ in range(40):
    time.sleep(2)
    logs = Path('/content/brandpulse_tunnel.log').read_text(errors='replace')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', logs)
    if match:
        public_url = match.group(0)
        break
print('Ссылка на приложение:', public_url or 'Пока не получена; см. /content/brandpulse_tunnel.log')


## Демонстрационный сценарий
1. В разделе «Сбор данных» загрузите демоданные или CSV.
2. Перейдите в Dashboard и выберите нужный набор и язык.
3. Покажите комментарии, PR-дайджест и «Сравнение моделей».
4. Для F1 загрузите ручную разметку по `comment_id` на странице «Сбор данных».

Для отдельного сравнения Qwen запустите `BrandPulse_Qwen_Comparison_Colab.ipynb`.